# 02 Spectral Analysis Pipeline

This notebook replaces the exploratory spectral notebooks with one reproducible pipeline.  It reads calibrated 1-D FITS spectra from `data/`, measures conservative sparse-spectrum diagnostics, writes CSV tables to `output/analysis_pipeline/`, and creates report-ready figures.

Literature grounding from `paper/sparse-multi-epoch-sn-spectra/`: with 1-3 spectra per object, robust claims should focus on type/subtype checks, spectral phase, velocities, pEW/FWHM, host contamination, and comparison to public samples.  TARDIS remains an interpretive aid, not the primary evidence.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.spectral_pipeline import build_all

ANALYSIS_DIR = PROJECT_ROOT / "output" / "analysis_pipeline"
FIG_DIR = ANALYSIS_DIR / "figures"

## Run or refresh the analysis products

In [ ]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    paths = build_all(PROJECT_ROOT)
    print(f"Updated {ANALYSIS_DIR}")
    for item in paths.get("figures", []):
        print(item)
else:
    print("Using existing output/analysis_pipeline products.")

## Target-level science status

In [ ]:
target_status = pd.read_csv(ANALYSIS_DIR / "target_status.csv")
display(target_status)

## Line diagnostics with quality flags

`qc_flag=adopt` means the automatic measurement passed conservative checks and is suitable for first-pass plots.  `qc_flag=check` should be inspected visually before any scientific claim.  This prevents over-interpreting noisy minima, broad blends, or secondary lines.

In [ ]:
line_qc = pd.read_csv(ANALYSIS_DIR / "line_diagnostics_qc.csv")
display(line_qc[["target", "date_obs", "phase_days", "type", "line", "velocity_kms", "pEW_A", "FWHM_A", "qc_flag", "qc_note"]])

## Host/environment diagnostics

In [ ]:
host_summary = pd.read_csv(ANALYSIS_DIR / "host_environment_summary.csv")
host_lines = pd.read_csv(ANALYSIS_DIR / "host_environment_lines.csv")
display(host_summary)
display(host_lines[host_lines["status"].eq("detected")].head(30))

## Report-ready figures

In [ ]:
for fig in [
    "target_status_table.png",
    "line_velocity_evolution.png",
    "pew_evolution.png",
    "blackbody_temperature.png",
    "host_line_detections.png",
]:
    path = FIG_DIR / fig
    if path.exists():
        display(Markdown(f"### {fig}"))
        display(Image(filename=str(path)))

## Spectral sequences

In [ ]:
for path in sorted(FIG_DIR.glob("spectral_sequence_*.png")):
    display(Markdown(f"### {path.stem.replace('_', ' ')}"))
    display(Image(filename=str(path)))